# Make YOLOv9t DPU-ready: Hardswish + pruning + fine-tuning (Colab GPU)

Structured channel pruning of the pipeline's detector, then fine-tuning on COCO to recover accuracy.
Code: [`scripts/prune_yolo.py`](https://github.com/shadowPunch/Optic-Flow-Estimation/blob/main/scripts/prune_yolo.py),
[`collision_avoidance/yolo_pruning.py`](https://github.com/shadowPunch/Optic-Flow-Estimation/blob/main/collision_avoidance/yolo_pruning.py).

**Before running:** *Runtime -> Change runtime type -> GPU* (T4 or better).

What it does (every step is logged to Weights & Biases, project `kria-collision-avoidance`):
1. evaluates the original YOLOv9t on COCO val2017,
2. swaps SiLU for Hardswish (the KV260 DPU cannot run SiLU) and prunes 30 % of channels
   (about -46 % GFLOPs; the Detect head is kept intact). With both changes the Vitis AI inspector
   places every operator on the DPU,
3. evaluates the modified model (accuracy collapses before fine-tuning; that is expected),
4. fine-tunes on a quarter of COCO train2017, evaluates again,
5. saves `yolov9t_pruned.pt` to `MyDrive/kria-collision-avoidance/`.

Rough cost on a T4: ~20 GB COCO download (10-20 min) plus ~3 h of training for the defaults.
Checkpoints are written to Drive after each epoch, so a disconnect does not lose finished epochs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
!git clone -q https://github.com/shadowPunch/Optic-Flow-Estimation.git /content/repo
%cd /content/repo
# Colab's own torch is used; only the extras are installed.
!pip install -q "ultralytics==8.3.253" "torch-pruning==1.6.1" wandb

In [ ]:
import wandb
# Preferred: add your key under the key icon (Secrets) in the left sidebar as WANDB_API_KEY
# and enable notebook access. Otherwise wandb asks: answer 2, then paste the key.
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
except Exception:
    key = None
wandb.login(key=key)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kria-collision-avoidance"
!mkdir -p "$OUT"

## Configuration
`RATIO` is the fraction of channels removed per layer (0.3 -> about -46 % GFLOPs). More `EPOCHS` / a larger
`FRACTION` recover more accuracy at the cost of time.

In [ ]:
RATIO = 0.3
EPOCHS = 20
FRACTION = 0.25   # share of COCO train2017 used for fine-tuning
BATCH = 64        # lower to 32 if the GPU runs out of memory

## Prune and fine-tune
Ultralytics downloads COCO automatically on first use.

In [ ]:
!python scripts/prune_yolo.py --data coco.yaml --ratio $RATIO --epochs $EPOCHS --fraction $FRACTION \
    --batch $BATCH --device 0 --workers 2 --project "$OUT/runs" --out "$OUT/yolov9t_pruned.pt"

## Result
`yolov9t_pruned.pt` is now in `MyDrive/kria-collision-avoidance/`. Back on the development machine, copy it to
`outputs/models/yolov9t_pruned.pt`; `deploy/vitis_ai/quantize.py --model yolov9t --yolo-weights ...` quantizes it
for the KV260 like the unpruned model. The W&B run (job type `prune`) holds original / pruned / fine-tuned mAP and GFLOPs.

In [ ]:
!ls -la "$OUT"